# Entrenamiento de Random Forest - Predicción de Magnitud Sísmica

**Grupo 5:**
- Israel López - Data Lead
- Jaime Sarabia - Product Lead  
- Diego Valdivieso - Model Lead

---

## 1. Importación de Librerías y Configuración

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 2. Carga del Dataset Procesado

In [ ]:
# Obtener la ruta base del proyecto
base = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
path = os.path.join(base, "01_datos_procesados", "sismos_procesados.parquet")

# Cargar datos procesados
df = pd.read_parquet(path)
print(f"Dataset cargado correctamente. Dimensiones: {df.shape}")
df.head()

## 3. Preparación de Variables (Feature Engineering)

In [ ]:
# Seleccionar características predictoras (Features) y variable objetivo (Target)
# Basados en el análisis exploratorio, usaremos:
# - latitud (lat)
# - longitud (lon)
# - profundidad (depth)
# También podríamos extraer componentes temporales adicionales si se considera necesario

X = df[['lat', 'lon', 'depth']]
y = df['magnitude']

print("Características predictoras (X):")
print(X.head())
print("\nVariable objetivo (y):")
print(y.head())

## 4. División del Dataset en Entrenamiento y Prueba

In [ ]:
# División estándar 80% entrenamiento y 20% prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Muestras de entrenamiento: {X_train.shape[0]}")
print(f"Muestras de prueba: {X_test.shape[0]}")

## 5. Entrenamiento del Modelo Random Forest Regressor

In [ ]:
# Inicializar el regresor con hiperparámetros base
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# Entrenar el modelo
print("Entrenando el modelo Random Forest...")
rf_model.fit(X_train, y_train)
print("¡Modelo entrenado con éxito!")

## 6. Evaluación del Modelo

In [ ]:
# Realizar predicciones sobre el conjunto de prueba
y_pred = rf_model.predict(X_test)

# Calcular métricas de evaluación
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Evaluar el KPI: Error Absoluto Medio porcentual promedio respecto a la magnitud real
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("=== MÉTRICAS DE RENDIMIENTO ===")
print(f"Error Absoluto Medio (MAE): {mae:.4f}")
print(f"Raíz del Error Cuadrático Medio (RMSE): {rmse:.4f}")
print(f"Coeficiente de Determinación (R²): {r2:.4f}")
print(f"Error Porcentual Absoluto Medio (MAPE): {mape:.2f}%")

# Verificar si se cumple el KPI (< 15% de error relativo)
if mape < 15.0:
    print("\n✅ ¡El modelo cumple con la métrica de éxito! (MAPE < 15%)")
else:
    print("\n❌ El modelo supera el límite del 15% de error porcentual definido en los KPIs.")

## 7. Optimización de Hiperparámetros (Búsqueda por Rejilla)

In [ ]:
# Definir la rejilla de parámetros a explorar
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# Configurar GridSearchCV con validación cruzada de 3 pliegues
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_absolute_error',
    verbose=1
)

print("Iniciando la búsqueda del mejor modelo...")
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
print(f"Mejores parámetros encontrados: {grid_search.best_params_}")

## 8. Evaluación del Mejor Modelo

In [ ]:
# Evaluar modelo optimizado
y_pred_opt = best_rf.predict(X_test)
mae_opt = mean_absolute_error(y_test, y_pred_opt)
rmse_opt = np.sqrt(mean_squared_error(y_test, y_pred_opt))
r2_opt = r2_score(y_test, y_pred_opt)
mape_opt = np.mean(np.abs((y_test - y_pred_opt) / y_test)) * 100

print("=== MÉTRICAS DEL MODELO OPTIMIZADO ===")
print(f"Error Absoluto Medio (MAE): {mae_opt:.4f}")
print(f"Raíz del Error Cuadrático Medio (RMSE): {rmse_opt:.4f}")
print(f"Coeficiente de Determinación (R²): {r2_opt:.4f}")
print(f"Error Porcentual Absoluto Medio (MAPE): {mape_opt:.2f}%")

## 9. Visualización de Resultados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Importancia de las variables (Feature Importance)
importances = best_rf.feature_importances_
indices = np.argsort(importances)
features = X.columns

axes[0].barh(range(len(indices)), importances[indices], color='steelblue', align='center')
axes[0].set_yticks(range(len(indices)))
axes[0].set_yticklabels([features[i] for i in indices])
axes[0].set_xlabel('Importancia Relativa')
axes[0].set_title('Importancia de Variables (Random Forest)')

# 2. Predicciones vs Valores Reales
axes[1].scatter(y_test, y_pred_opt, alpha=0.3, color='darkred', edgecolors='black', linewidth=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2, label='Ideal')
axes[1].set_xlabel('Magnitud Real')
axes[1].set_ylabel('Magnitud Predicha')
axes[1].set_title('Valores Predichos vs Reales')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Guardado del Modelo Entrenado

In [ ]:
# Guardar el modelo entrenado y los metadatos en la carpeta de modelos
model_export_path = os.path.join(base, "04_modelos", "random_forest_regressor.joblib")

joblib.dump(best_rf, model_export_path)
print(f"Modelo Random Forest exportado exitosamente en: {model_export_path}")